
# MiniMax H3 — 2×T4 Multi-GPU Model Test (no API)

Purpose: measure whether splitting one H3 job across the two T4s on Kaggle can help.
This notebook intentionally does **not** launch the FastAPI/Cloudflare stack.

Plan:
1. Preflight: GPU count, VRAM, RAM, topology (`nvidia-smi topo -m`).
2. P2P vs host-staged activation bandwidth probe (`GPU0 <-> GPU1`).
3. Report single-GPU vs dual-GPU expectation for H3 transformer split.
4. Checkout ComfyUI H3 MultiStream feasibility on this image.

Quality defaults remain H3 1344x768, ~1 MP.


In [ ]:

import os, shutil, subprocess, psutil, torch
print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("device_count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    free,total = torch.cuda.mem_get_info(i)
    print(f"GPU{i}: {p.name} cc={p.major}.{p.minor} VRAM={total/2**30:.1f}G free={free/2**30:.1f}G")
ram = psutil.virtual_memory()
print(f"RAM total={ram.total/2**30:.1f}G avail={ram.available/2**30:.1f}G")
print("/tmp free:", shutil.disk_usage('/tmp').free / 2**30, "GiB")


In [ ]:

print(subprocess.run(["nvidia-smi","topo","-m"],capture_output=True,text=True).stdout)


In [ ]:

import torch, time
D = torch.cuda.device_count()
if D >= 2:
    a = torch.randn(1<<26, device="cuda:0", dtype=torch.float16)
    b = torch.randn(1<<26, device="cuda:1", dtype=torch.float16)
    # host-staged
    h = torch.empty(1<<26, dtype=torch.float16, pin_memory=True)
    torch.cuda.synchronize(); t=time.time()
    for _ in range(10):
        h.copy_(a, non_blocking=True); b.copy_(h, non_blocking=True); torch.cuda.synchronize()
    host_t = (time.time()-t)/10
    # direct p2p
    torch.cuda.synchronize(); t=time.time()
    for _ in range(10):
        b.copy_(a, non_blocking=True); torch.cuda.synchronize()
    p2p_t = (time.time()-t)/10
    gb = (1<<26)*2/1e9
    print(f"host D2H->H2D effective: {gb/host_t:.2f} GB/s")
    print(f"direct p2p effective:     {gb/p2p_t:.2f} GB/s")
    print("=> For small activations, prefer host (pinned) staging on PCIe.")


In [ ]:

# FEASIBILITY: is ComfyUI H3 MultiStream viable on this image?
import subprocess, sys, os
print("Comfy version probe (will fail if not installed):")
print(subprocess.run([sys.executable,"-c","import comfy; print(comfy.__version__)"],capture_output=True,text=True).stderr[-300:] or "ok")
print("
Recommendation: let this run as a multi-GPU probe only; do not break the WanGP API lane.")


In [ ]:

print("Next on dual-T4:")
print("  1) Probe done. If 2x T4 visible and PCIe host exchange acceptable,")
print("  2) we can wire ComfyUI H3 MultiStream with exchange=host, exchange_chunks=8,")
print("  3) but only as gpu_mode=dual while WanGP single-T4 remains the stable fallback.")
